# Biohub Detector Fine-Tuning
Fine-tunes the UNet3D detection head on the 8 training movies.
Loads pretrained weights from the support pack, trains on .geff ground truth.

**Setup:** Attach `biohub-tracking-support-pack-50ep-v1` + competition data. GPU T4 x2. Internet OFF. Run All.
Output: `/kaggle/working/finetuned_detector.pt`

In [ ]:
import os, sys, json, re
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import subprocess

# Install zarr from offline wheels if missing (internet is OFF)
try:
    import zarr
    print("zarr already available", flush=True)
except ImportError:
    print("Installing zarr from offline wheels...", flush=True)
    # Find wheels dir in support pack (need SUPPORT first, so do minimal search)
    import pathlib
    sp = None
    for p in list(pathlib.Path('/kaggle/input/datasets').rglob('biohub-tracking-support-pack-50ep-v1')):
        sp = p
        break
    if sp is None:
        for p in list(pathlib.Path('/kaggle/input').rglob('biohub-tracking-support-pack-50ep-v1')):
            sp = p
            break
    assert sp is not None, "Support pack not found for wheels"
    wheels = sp / 'wheels'
    assert wheels.exists(), f"Wheels dir not found: {wheels}"
    cmd = [sys.executable, '-m', 'pip', 'install', '--no-deps', '--find-links', str(wheels), 'zarr']
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout[-500:] if result.stdout else "", flush=True)
    if result.returncode != 0:
        print(result.stderr[-1000:] if result.stderr else "", flush=True)
        raise RuntimeError("zarr install failed")
    import zarr
    print("zarr installed", flush=True)

WORKING = Path('/kaggle/working')
COMP = Path('/kaggle/input/biohub-cell-tracking-during-development')
TRAIN_DIR = COMP / 'train'

# Find support pack (nested mount aware)
candidates = list(Path('/kaggle/input/datasets').rglob('biohub-tracking-support-pack-50ep-v1'))
if not candidates:
    candidates = list(Path('/kaggle/input').rglob('biohub-tracking-support-pack-50ep-v1'))
assert candidates, "Support pack not found!"
SUPPORT = candidates[0]
print(f"Support pack: {SUPPORT}", flush=True)

# Explore structure
print("Top-level contents:", flush=True)
for p in sorted(SUPPORT.iterdir())[:20]:
    print(f"  {p.name} ({'dir' if p.is_dir() else 'file'})", flush=True)

# Find the model code: search for temporal_unet.py or biohub_tracking
model_py = list(SUPPORT.rglob('temporal_unet.py'))
if not model_py:
    model_py = list(SUPPORT.rglob('biohub_tracking'))
print(f"Model files found: {len(model_py)}", flush=True)
for p in model_py[:5]:
    print(f"  {p}", flush=True)

# Determine src path: parent of biohub_tracking, or SUPPORT/src
src_path = None
if model_py:
    mp = model_py[0]
    if mp.name == 'temporal_unet.py':
        # .../biohub_tracking/models/temporal_unet.py -> src is 3 levels up
        src_path = mp.parent.parent.parent
    elif mp.name == 'biohub_tracking':
        src_path = mp.parent
    print(f"Derived src_path: {src_path}", flush=True)

if src_path is None or not src_path.exists():
    # Fallback: try SUPPORT/src
    cand = SUPPORT / 'src'
    if cand.exists():
        src_path = cand
        print(f"Fallback src_path: {src_path}", flush=True)

assert src_path is not None and src_path.exists(), f"Could not locate model src in {SUPPORT}"
sys.path.insert(0, str(src_path))
print(f"Added to path: {src_path}", flush=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}", flush=True)


In [ ]:
# Adaptively find the UNet model class (uses model_py from setup)
import importlib
import importlib.util

# model_py was found in setup; get the temporal_unet.py path
model_file = None
for p in model_py:
    if p.name == 'temporal_unet.py':
        model_file = p
        break
if model_file is None:
    # Search from src_path
    found = list(src_path.rglob('temporal_unet.py'))
    assert found, "temporal_unet.py not found!"
    model_file = found[0]

print(f"Model file: {model_file}", flush=True)
assert model_file.exists(), f"Model file not found: {model_file}"

text = model_file.read_text()
classes = re.findall(r'^class\s+(\w+)', text, re.MULTILINE)
print(f"Classes in temporal_unet.py: {classes}", flush=True)

spec = importlib.util.spec_from_file_location("temporal_unet", str(model_file))
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

UNetClass = None
for cls_name in classes:
    if 'unet' in cls_name.lower():
        UNetClass = getattr(mod, cls_name)
        print(f"Using UNet class: {cls_name}", flush=True)
        break
assert UNetClass is not None, f"No UNet class found in {classes}"


In [ ]:
# Find pretrained weights (any checkpoint format)
# Look in weights/ dir first, then anywhere
weights_dir = SUPPORT / 'weights'
ckpt_files = []
if weights_dir.exists():
    print(f"Weights dir contents:", flush=True)
    for p in sorted(weights_dir.rglob('*')):
        if p.is_file():
            print(f"  {p.relative_to(SUPPORT)} ({p.stat().st_size/1e6:.1f} MB)", flush=True)
            if p.suffix in ['.pt', '.pth', '.bin', '.safetensors', '.ckpt']:
                ckpt_files.append(p)

if not ckpt_files:
    # Fallback: search entire support pack
    for ext in ['*.pt', '*.pth', '*.bin', '*.safetensors', '*.ckpt']:
        ckpt_files.extend(SUPPORT.rglob(ext))
    print(f"Fallback search found {len(ckpt_files)} checkpoint files", flush=True)

print(f"Found {len(ckpt_files)} checkpoint files", flush=True)
for f in ckpt_files[:10]:
    print(f"  {f.relative_to(SUPPORT)} ({f.stat().st_size/1e6:.1f} MB)")

# Pick the primary detector weights (largest, excluding deepcenter/secondary)
weights_path = None
for f in ckpt_files:
    name = f.name.lower()
    if 'deepcenter' not in name and 'secondary' not in name:
        if weights_path is None or f.stat().st_size > weights_path.stat().st_size:
            weights_path = f

assert weights_path is not None, "No suitable weights found!"
print(f"Using weights: {weights_path}", flush=True)


In [ ]:
# Instantiate model and load weights
model = None
for kwargs in [{}, {'in_channels': 1}, {'in_channels': 2}]:
    try:
        m = UNetClass(**kwargs)
        print(f"Instantiated with {kwargs}", flush=True)
        model = m
        break
    except Exception as e:
        print(f"Failed with {kwargs}: {e}", flush=True)
        continue
assert model is not None, "Could not instantiate model"

# Load weights (handle .pt/.pth/.bin and .safetensors)
wp = str(weights_path)
if wp.endswith('.safetensors'):
    try:
        from safetensors.torch import load_file
        sd = load_file(wp, device='cpu')
        print("Loaded via safetensors", flush=True)
    except ImportError:
        raise RuntimeError("safetensors not installed but weights are .safetensors")
else:
    sd = torch.load(wp, map_location='cpu', weights_only=False)
    print("Loaded via torch.load", flush=True)

if isinstance(sd, dict):
    # Checkpoint dict: extract the actual state dict
    for key in ['model_state_dict', 'state_dict', 'model']:
        if key in sd:
            sd = sd[key]
            print(f"Extracted '{key}' from checkpoint", flush=True)
            break
sd = {k.replace('module.', ''): v for k, v in sd.items()}

try:
    model.load_state_dict(sd, strict=True)
    print("Loaded weights (strict)", flush=True)
except Exception as e:
    print(f"Strict load failed: {e}", flush=True)
    model.load_state_dict(sd, strict=False)
    print("Loaded weights (non-strict)", flush=True)

model = model.to(device)
model.train()
print(f"Model on {device}, train mode", flush=True)


In [ ]:
# Freeze encoder, fine-tune decoder/head only
# Heuristic: freeze params with 'encoder'/'down'/'contract' in name
frozen = 0
trainable = 0
for name, param in model.named_parameters():
    n = name.lower()
    if any(k in n for k in ['encoder', 'down', 'contract', 'bottleneck']):
        param.requires_grad = False
        frozen += param.numel()
    else:
        param.requires_grad = True
        trainable += param.numel()

print(f"Frozen: {frozen:,} params, Trainable: {trainable:,} params", flush=True)

# If nothing trainable (naming mismatch), unfreeze all with low LR
if trainable == 0:
    print("WARNING: no params matched, unfreezing all", flush=True)
    for p in model.parameters():
        p.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters())

optimizer = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4
)
criterion = nn.BCEWithLogitsLoss()
print("Optimizer ready", flush=True)


In [ ]:
# Load ground truth centroids from all 8 .geff files
import zarr

geff_files = sorted(TRAIN_DIR.glob('*.geff'))
print(f"Found {len(geff_files)} .geff files", flush=True)

# Voxel spacing (um/voxel): x=0.40625, y=0.40625, z=1.625, z_offset=7.0
# Convert microns to voxels
def um_to_voxel(x_um, y_um, z_um):
    vx = x_um / 0.40625
    vy = y_um / 0.40625
    vz = (z_um - 7.0) / 1.625
    return vx, vy, vz

# Load all centroids: dict movie_name -> list of (t, vx, vy, vz)
all_gt = {}
for gf in geff_files:
    name = gf.stem
    store = zarr.open(str(gf), mode='r')
    props = store['nodes']['props']
    t = props['t']['values'][:]
    x = props['x']['values'][:]
    y = props['y']['values'][:]
    z = props['z']['values'][:]
    cells = []
    for i in range(len(t)):
        vx, vy, vz = um_to_voxel(x[i], y[i], z[i])
        cells.append((int(t[i]), vx, vy, vz))
    all_gt[name] = cells
    print(f"  {name}: {len(cells)} cells", flush=True)

print(f"\nTotal movies: {len(all_gt)}", flush=True)


In [ ]:
# Create Gaussian heatmap target from centroids
def make_heatmap(centroids, shape=(64, 64, 64), sigma=2.0):
    """centroids: list of (vz, vy, vx) in voxel coords within the patch"""
    zz, yy, xx = np.meshgrid(
        np.arange(shape[0]), np.arange(shape[1]), np.arange(shape[2]),
        indexing='ij'
    )
    heatmap = np.zeros(shape, dtype=np.float32)
    for (cz, cy, cx) in centroids:
        # Gaussian centered at (cz, cy, cx)
        g = np.exp(-((zz - cz)**2 + (yy - cy)**2 + (xx - cx)**2) / (2 * sigma**2))
        heatmap = np.maximum(heatmap, g)
    return heatmap

# Test
test_hm = make_heatmap([(32, 32, 32), (10, 20, 30)])
print(f"Heatmap shape: {test_hm.shape}, max: {test_hm.max():.3f}", flush=True)
assert test_hm.shape == (64, 64, 64)
assert abs(test_hm.max() - 1.0) < 0.01
print("PASS: heatmap generation", flush=True)


In [ ]:
# Training loop
# For each movie, for each sampled frame: extract 64^3 patch, create target, train

PATCH = 64
EPOCHS = 3
FRAMES_PER_MOVIE = 20  # sample to save time
SIGMA = 2.0

# Find movie zarr paths
movie_zarrs = {}
for name in all_gt.keys():
    # Movie zarr is in train/ with matching name
    for ext in ['', '.zarr']:
        p = TRAIN_DIR / (name + ext)
        if p.exists():
            movie_zarrs[name] = p
            break
print(f"Found {len(movie_zarrs)} movie zarrs", flush=True)
for name, p in list(movie_zarrs.items())[:3]:
    print(f"  {name}: {p}")

# Training
model.train()
for epoch in range(EPOCHS):
    total_loss = 0.0
    n_batches = 0
    for movie_name, cells in all_gt.items():
        if movie_name not in movie_zarrs:
            continue
        zarr_path = movie_zarrs[movie_name]
        try:
            store = zarr.open(str(zarr_path / '0'), mode='r')
        except:
            try:
                store = zarr.open(str(zarr_path), mode='r')
            except Exception as e:
                print(f"  Skip {movie_name}: {e}", flush=True)
                continue
        
        # Group cells by frame
        by_frame = {}
        for (t, vx, vy, vz) in cells:
            by_frame.setdefault(t, []).append((vz, vy, vx))
        
        # Sample frames
        frames = sorted(by_frame.keys())
        if len(frames) > FRAMES_PER_MOVIE:
            idx = np.linspace(0, len(frames)-1, FRAMES_PER_MOVIE, dtype=int)
            frames = [frames[i] for i in idx]
        
        for t in frames:
            try:
                # Load frame (assume shape is (Z, Y, X) or (C, Z, Y, X))
                vol = store[t]
                if vol.ndim == 4:
                    vol = vol[0]  # take first channel
                # vol shape should be (Z, Y, X)
                Z, Y, X = vol.shape
                
                # Random 64^3 crop
                z0 = np.random.randint(0, max(1, Z - PATCH))
                y0 = np.random.randint(0, max(1, Y - PATCH))
                x0 = np.random.randint(0, max(1, X - PATCH))
                patch = vol[z0:z0+PATCH, y0:y0+PATCH, x0:x0+PATCH]
                
                # Pad if needed
                if patch.shape != (PATCH, PATCH, PATCH):
                    pad_z = PATCH - patch.shape[0]
                    pad_y = PATCH - patch.shape[1]
                    pad_x = PATCH - patch.shape[2]
                    patch = np.pad(patch, ((0, pad_z), (0, pad_y), (0, pad_x)))
                
                # Find centroids in this patch
                patch_cells = []
                for (cz, cy, cx) in by_frame[t]:
                    pz, py, px = cz - z0, cy - y0, cx - x0
                    if 0 <= pz < PATCH and 0 <= py < PATCH and 0 <= px < PATCH:
                        patch_cells.append((pz, py, px))
                
                # Create target heatmap
                target = make_heatmap(patch_cells, sigma=SIGMA)
                
                # Normalize input (heuristic: clip and scale)
                inp = patch.astype(np.float32)
                inp = np.clip(inp, 0, np.percentile(inp, 99.9))
                inp = inp / (inp.max() + 1e-6)
                
                # Model expects (B, C, D, H, W) or (B, T, D, H, W)?
                # Try (1, 1, 64, 64, 64) - single frame, single channel
                inp_t = torch.from_numpy(inp).unsqueeze(0).unsqueeze(0).to(device)
                tgt_t = torch.from_numpy(target).unsqueeze(0).unsqueeze(0).to(device)
                
                # Forward - try model(inp) or model.encode(inp)
                try:
                    out = model(inp_t)
                    if isinstance(out, tuple):
                        # Take the detection output (likely last or named)
                        logits = out[-1] if len(out) > 1 else out[0]
                    else:
                        logits = out
                except Exception as e:
                    print(f"  Forward failed: {e}", flush=True)
                    continue
                
                # Ensure shapes match
                if logits.shape != tgt_t.shape:
                    print(f"  Shape mismatch: {logits.shape} vs {tgt_t.shape}, skipping", flush=True)
                    continue
                
                loss = criterion(logits, tgt_t)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
                n_batches += 1
                
            except Exception as e:
                continue
    
    avg_loss = total_loss / max(1, n_batches)
    print(f"Epoch {epoch+1}/{EPOCHS}: loss={avg_loss:.4f} ({n_batches} batches)", flush=True)

print("Training complete", flush=True)


In [ ]:
# Save fine-tuned weights
out_path = WORKING / 'finetuned_detector.pt'
torch.save(model.state_dict(), str(out_path))
print(f"Saved to {out_path} ({out_path.stat().st_size/1e6:.1f} MB)", flush=True)

# Quick validation: run inference on a dummy input
model.eval()
with torch.no_grad():
    dummy = torch.randn(1, 1, 64, 64, 64).to(device)
    try:
        out = model(dummy)
        if isinstance(out, tuple):
            out = out[-1]
        print(f"Validation forward: input {dummy.shape} -> output {out.shape}", flush=True)
        print("PASS: model works", flush=True)
    except Exception as e:
        print(f"Validation failed: {e}", flush=True)
